# Prompt 1 — What is Infrastructure as Code (IaC)?
## Terraform Associate (004) Exam Study Guide

**Exam Objective 1**: Understand what IaC is, its advantages, and how Terraform fits the multi-cloud landscape.

---

**Topics covered in this notebook:**
1. IaC Definition
2. Core IaC Principles — Idempotency, Declarative vs Imperative
3. Advantages of IaC over Manual Provisioning
4. Where Terraform Fits — Declarative Tooling
5. IaC and Team Collaboration
6. Terraform's Desired-State Approach
7. Multi-Cloud and Service-Agnostic Advantages
8. Hybrid Cloud Use Cases
9. Real-World Rebuild Scenario
10. Exam-Style Questions (3)
11. Real-World Scenarios (5)

---
## 1. What is Infrastructure as Code?

**Infrastructure as Code (IaC)** is the practice of managing and provisioning computing infrastructure through machine-readable configuration files rather than through manual processes, graphical user interfaces, or interactive command-line sessions.

Instead of a sysadmin clicking through an AWS console to create a VPC, subnets, security groups, and EC2 instances, an engineer writes a `.tf` file that describes the desired end state and commits it to Git. The tool (Terraform) reads that file and handles all API calls automatically.

### Key Definition Points (Exam Language)

| Term | Meaning |
|------|---------|
| **Machine-readable** | Configuration files (HCL, JSON, YAML) that tools can parse and execute |
| **Provisioning** | Creating and configuring infrastructure resources |
| **Configuration management** | Maintaining desired state of existing resources over time |
| **Desired state** | The infrastructure you want to exist — declared in code |
| **Current state** | What actually exists right now — tracked in state file |

### IaC vs Manual Provisioning — Side by Side

| Dimension | Manual / ClickOps | Infrastructure as Code |
|-----------|------------------|-----------------------|
| Repeatability | Error-prone; hard to reproduce | Identical result every run |
| Documentation | In someone's head or stale wiki | The code IS the documentation |
| Speed | Hours to days for complex environments | Minutes once code exists |
| Audit trail | Browser history, memory | Full Git history with author, date, diff |
| Disaster recovery | Manual rebuilds from notes | `terraform apply` — rebuild in minutes |
| Drift detection | No visibility | `terraform plan` shows drift immediately |

---
## 2. Core IaC Principles

### 2.1 Idempotency

**Idempotency** means you can apply the same configuration multiple times and always get the same result — running it once or a hundred times produces identical infrastructure.

- **Without IaC**: Running a shell script twice may create duplicate resources or fail on the second run.
- **With Terraform**: Running `terraform apply` on an already-applied configuration results in `No changes. Your infrastructure matches the configuration.`

Terraform achieves idempotency by comparing the **desired state** (your `.tf` files) against the **current state** (`.terraform.tfstate`) and the **actual state** (live API calls) before making any changes.

```
Run 1: 0 resources exist → Terraform creates 5 resources
Run 2: 5 resources exist matching config → Terraform makes 0 changes  ← idempotent
Run 3: Someone manually adds a tag → Terraform removes the extra tag  ← drift correction
```

### 2.2 Declarative vs Imperative Approaches

This is a key exam distinction.

#### Declarative (What, not How)
You describe the **desired end state**. The tool figures out what steps to take.

```hcl
# Terraform HCL — declarative
# "I want 3 EC2 instances of type t3.micro in us-east-1"
resource "aws_instance" "web" {
  count         = 3
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
}
```

Terraform determines: create 3 instances, wire dependencies, handle ordering.

#### Imperative (How to get there)
You describe the **sequence of steps** to execute. The tool follows your instructions literally.

```bash
# Bash script — imperative
# "Do these steps in this order"
aws ec2 run-instances --image-id ami-0c55b159cbfafe1f0 --instance-type t3.micro
aws ec2 run-instances --image-id ami-0c55b159cbfafe1f0 --instance-type t3.micro
aws ec2 run-instances --image-id ami-0c55b159cbfafe1f0 --instance-type t3.micro
# Run this twice → 6 instances (not idempotent!)
```

| Approach | Tools | Exam Key Point |
|----------|-------|---------------|
| **Declarative** | Terraform, AWS CloudFormation, Kubernetes manifests | Describe the desired state; tool handles the diff |
| **Imperative** | Ansible playbooks (partially), Bash scripts, Chef recipes | Describe the steps; tool executes them in order |

> **Exam tip**: Terraform is **declarative**. Ansible is often described as **imperative** (though it has idempotent modules). This distinction appears frequently on the exam.

---
## 3. Advantages of IaC over Manual Provisioning

The exam tests whether you can identify and explain these advantages.

### 3.1 Consistency and Repeatability
The same Terraform configuration applied to dev, staging, and production produces **identical resource configurations** — no environment-specific surprises caused by different engineers clicking different options.

### 3.2 Version Control
Infrastructure code lives in Git alongside application code:
- Every change has an author, timestamp, and commit message
- You can `git diff` to see exactly what changed between deployments
- You can `git revert` to roll back a bad infrastructure change
- Branch protection rules enforce peer review before infrastructure changes reach production

### 3.3 Peer Review
Infrastructure changes go through pull requests, enabling:
- Security team review before new IAM policies go live
- Cost review before expensive instance types are added
- Compliance checks (no public S3 buckets, encryption required, etc.)

### 3.4 Automation
IaC integrates with CI/CD pipelines:
```
PR opened  → terraform fmt -check && terraform validate
PR approved → terraform plan (output posted as PR comment)
Merge to main → terraform apply -auto-approve
```

### 3.5 Documentation as Code
The `.tf` files **are** the documentation. They always reflect what exists (unlike a wiki page that goes stale). Engineers joining a team can read the Terraform code to understand the entire infrastructure topology.

### 3.6 Disaster Recovery Speed
Manual approach: Rebuild a multi-tier application environment from notes and memory — could take days.

IaC approach:
```bash
git clone https://github.com/company/infrastructure
cd infrastructure/production
terraform init
terraform apply -auto-approve
# Environment rebuilt in 15 minutes
```

### 3.7 Cost Management
- `terraform plan` shows resource additions before they cost money
- Tools like Infracost integrate with Terraform to show cost estimates per PR
- `terraform destroy` cleanly removes an entire test environment — no forgotten resources running up costs

### Summary Table

| Advantage | How IaC Delivers It |
|-----------|--------------------|
| Consistency | Same code → same result every environment |
| Repeatability | Idempotent `apply` — safe to run multiple times |
| Version control | Git tracks all infrastructure changes |
| Peer review | PRs gate infrastructure changes |
| Automation | CI/CD pipelines run plan/apply |
| Documentation | Code is the source of truth |
| Disaster recovery | Rebuild with `terraform apply` in minutes |
| Cost visibility | Plan shows resource additions before apply |

---
## 4. Where Terraform Fits — Declarative IaC Tooling

### The IaC Tool Landscape

| Tool | Approach | Primary Use Case | Scope |
|------|----------|-----------------|-------|
| **Terraform** | Declarative | Provision infrastructure across any cloud/service | Infrastructure provisioning |
| **AWS CloudFormation** | Declarative | Provision AWS resources only | Infrastructure provisioning (AWS-only) |
| **Pulumi** | Imperative (general-purpose languages) | Infrastructure provisioning | Multi-cloud |
| **Ansible** | Imperative/Declarative hybrid | Configuration management, application deployment | Existing servers |
| **Chef / Puppet** | Declarative DSL | Configuration management on servers | Existing servers |
| **Bash / Python scripts** | Imperative | Ad-hoc automation | Any |

### Terraform's Position

Terraform is the industry-standard **declarative, cloud-agnostic infrastructure provisioning** tool:

- **Provisioning** (not configuration management): Terraform creates and manages cloud resources — VMs, networks, databases, DNS records, IAM policies. It does not install software on servers (use Ansible, Chef, or cloud-init for that).
- **Declarative**: You write HCL describing the desired end state. Terraform's planning engine (`Terraform Core`) computes what changes are needed.
- **Cloud-agnostic**: The same `terraform plan / apply` workflow works for AWS, Azure, GCP, Kubernetes, GitHub, Datadog, and 3,000+ other providers.

### Terraform vs Ansible — Common Exam Comparison

| | Terraform | Ansible |
|-|-----------|--------|
| Primary purpose | Provision infrastructure | Configure existing systems |
| Approach | Declarative (desired state) | Imperative (ordered playbooks) |
| State tracking | Yes — `terraform.tfstate` | No built-in state |
| Cloud-agnostic | Yes (3,000+ providers) | Yes (but via modules) |
| Idempotency | Built-in (state-based) | Module-dependent (not guaranteed) |
| Agent required | No (API-based) | No (agentless via SSH) |

> **Exam tip**: The most common IaC comparison question is Terraform vs Ansible. Terraform = **provision** (create/destroy resources). Ansible = **configure** (install packages, edit files on existing servers).

---
## 5. IaC Enables Team Collaboration

### Storing Configuration in Git

A typical repository structure for IaC:

```
infrastructure/
├── modules/
│   ├── vpc/          # reusable VPC module
│   ├── eks-cluster/  # reusable Kubernetes cluster module
│   └── rds/          # reusable database module
├── environments/
│   ├── dev/          # dev environment root module
│   ├── staging/
│   └── production/
└── .github/
    └── workflows/    # CI/CD pipeline for plan/apply
```

### Code Review Workflow

```
1. Engineer creates feature branch: git checkout -b add-redis-cluster
2. Writes Terraform code for ElastiCache Redis
3. Opens pull request → CI runs:
   - terraform fmt -check
   - terraform validate
   - terraform plan (output posted as PR comment)
4. Reviewers examine the plan diff — see exactly what will be created
5. Approved → merge to main → CD pipeline runs:
   - terraform apply -auto-approve
6. Resources created in production; Git history records who approved it
```

### Branching Strategies for Infrastructure

| Strategy | Description | Best For |
|----------|-------------|----------|
| **GitOps** | Merge to branch = deploy to environment | Automated, policy-driven teams |
| **Trunk-based** | Short-lived branches, fast merges to main | High-velocity teams |
| **Environment branches** | `dev`, `staging`, `prod` branches | Teams that want explicit environment promotion |

> **Exam tip**: Terraform itself does not enforce a branching strategy — that is a team convention. Terraform workspaces provide a mechanism for separate state per environment, but branching strategy is a process decision.

---
## 6. Terraform's Desired-State Approach

### How the Planning Engine Works

Terraform's core workflow compares three things:

```
┌─────────────────────────┐
│  1. Desired State       │  ← Your .tf configuration files
│     (what you want)     │
└────────────┬────────────┘
             │ compare
┌────────────▼────────────┐
│  2. Current State       │  ← terraform.tfstate (last known state)
│     (what Terraform     │
│      last recorded)     │
└────────────┬────────────┘
             │ refresh (API calls)
┌────────────▼────────────┐
│  3. Actual State        │  ← Real infrastructure (live API)
│     (what really        │
│      exists now)        │
└─────────────────────────┘
             │
             ▼
     Execution Plan
  (+ create, ~ update, - destroy)
```

### Execution Plan Output Symbols

| Symbol | Meaning |
|--------|---------|
| `+` | Resource will be **created** |
| `-` | Resource will be **destroyed** |
| `~` | Resource will be **updated in-place** |
| `-/+` | Resource will be **destroyed and recreated** (replacement) |
| `<=` | Data source will be **read** |

### Example Plan Output

```hcl
# main.tf — simple EC2 instance
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  tags = {
    Name = "web-server"
  }
}
```

```bash
$ terraform plan

Terraform will perform the following actions:

  # aws_instance.web will be created
  + resource "aws_instance" "web" {
      + ami           = "ami-0c55b159cbfafe1f0"
      + instance_type = "t3.micro"
      + tags          = {
          + "Name" = "web-server"
        }
      # ... (other computed attributes)
    }

Plan: 1 to add, 0 to change, 0 to destroy.
```

> **Key exam insight**: The **plan** is a safe, read-only preview. No infrastructure changes occur until `terraform apply` is run and confirmed. A saved plan file (`-out=plan.tfplan`) guarantees that `apply` executes **exactly** what was planned — even if someone changes the `.tf` files between plan and apply.

---
## 7. Multi-Cloud and Service-Agnostic Advantages

### One Workflow for All Providers

Terraform's plugin architecture means the same CLI commands work for every provider:

```bash
terraform init     # Download AWS, Azure, GCP providers
terraform plan     # Works identically for all three
terraform apply    # Creates resources across all providers
```

### Multi-Cloud Example

A single Terraform workspace managing resources across AWS and Azure:

```hcl
terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
    azurerm = {
      source  = "hashicorp/azurerm"
      version = "~> 3.0"
    }
  }
}

provider "aws" {
  region = "us-east-1"
}

provider "azurerm" {
  features {}
}

# AWS resource
resource "aws_s3_bucket" "data_lake" {
  bucket = "company-data-lake"
}

# Azure resource — same workflow, same commands
resource "azurerm_resource_group" "main" {
  name     = "company-rg"
  location = "East US"
}
```

### Provider Ecosystem Scale

Terraform providers exist for:
- **Cloud IaaS**: AWS, Azure, GCP, Oracle Cloud, Alibaba Cloud
- **Kubernetes**: Deploy manifests, Helm charts
- **Databases**: MySQL, PostgreSQL, MongoDB Atlas
- **SaaS / Monitoring**: Datadog, PagerDuty, Grafana, New Relic
- **DNS / CDN**: Cloudflare, AWS Route 53, Fastly
- **Identity**: Okta, Auth0, HashiCorp Vault
- **Version Control**: GitHub, GitLab, Bitbucket (manage repos, teams, webhooks)
- **Networking**: Palo Alto Networks, F5, Cisco

### Key Business Advantages

| Advantage | Detail |
|-----------|--------|
| **Vendor portability** | Configuration is a thin abstraction over provider APIs — easier to move workloads |
| **Skills transferability** | Engineers learn HCL once, work across all providers |
| **Unified workflow** | One CI/CD pipeline pattern for all infrastructure |
| **Reduced tool sprawl** | One tool instead of CloudFormation + ARM templates + gcloud deployment-manager |
| **Consistent governance** | Same policy-as-code (Sentinel/OPA) applies across all providers |

---
## 8. Hybrid Cloud Use Cases

Terraform manages not just public cloud resources but on-premises infrastructure alongside cloud — in the same workspace.

### On-Premises + Public Cloud in One Workspace

```hcl
terraform {
  required_providers {
    vsphere = {
      source  = "hashicorp/vsphere"
      version = "~> 2.0"
    }
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

# On-premises VMware VM
provider "vsphere" {
  user           = var.vsphere_user
  password       = var.vsphere_password
  vsphere_server = "vcenter.company.internal"
}

resource "vsphere_virtual_machine" "app" {
  name             = "app-server-01"
  resource_pool_id = data.vsphere_resource_pool.pool.id
  datastore_id     = data.vsphere_datastore.datastore.id
  num_cpus         = 4
  memory           = 8192
}

# AWS cloud resource in same workspace
provider "aws" { region = "us-east-1" }

resource "aws_vpn_gateway" "hybrid_link" {
  vpc_id = aws_vpc.main.id
  tags   = { Name = "hybrid-vpn-gw" }
}
```

### Common Hybrid Scenarios

| Scenario | Providers Used |
|----------|---------------|
| VMware on-prem + AWS cloud bursting | `vsphere` + `aws` |
| On-prem Kubernetes + Azure AKS | `helm` + `azurerm` |
| Physical network gear + cloud VPCs | `junos`/`iosxe` + `aws`/`azurerm` |
| Corporate DNS + Route53 | `infoblox` + `aws` |
| On-prem Active Directory + Azure AD | `ad` + `azuread` |

### Why This Matters for the Exam

The exam tests understanding that Terraform's value proposition includes:
1. A **single consistent workflow** regardless of the target system
2. The ability to manage **dependencies between on-premises and cloud resources** in a single plan
3. The ability to **migrate incrementally** — manage existing on-prem resources alongside new cloud resources during a cloud migration project

---
## 9. Real-World Scenario: Rebuild an Environment with IaC

### Without IaC — The Manual Nightmare

**Scenario**: A company's production environment (VPC, 3 subnets, ALB, 6 EC2 instances, RDS, ElastiCache, Route53 records, IAM roles, S3 buckets) is destroyed by an accidental `aws ec2 terminate-instances --instance-ids all`.

**Manual recovery process**:
1. Find the old architecture diagram (probably outdated)
2. Recreate VPC and subnets from memory and notes — Day 1
3. Configure security groups, route tables, internet gateway — Day 1–2
4. Launch EC2 instances, install software, restore from backup — Day 2–3
5. Configure RDS, restore database, update connection strings — Day 3–4
6. Update DNS records, test, debug issues caused by configuration drift — Day 4–5
7. **Total: 4–5 days, likely incorrect in several places**

### With IaC — 15-Minute Recovery

```bash
# The entire environment is code in Git
git clone https://github.com/company/infrastructure
cd infrastructure/production

# Initialize Terraform (downloads providers)
terraform init

# Preview what will be created
terraform plan
# Output: Plan: 47 to add, 0 to change, 0 to destroy.

# Create everything
terraform apply -auto-approve
# ...
# Apply complete! Resources: 47 added, 0 changed, 0 destroyed.
# Total time: ~12 minutes
```

**Result**: Identical environment rebuilt in 12–15 minutes. Every resource has the exact same configuration as before. No guesswork, no drift.

### Why This Works

- Every resource configuration is committed to Git — **the code is the runbook**
- Provider credentials (stored securely in a vault) are the only external dependency
- The Git history shows exactly what the environment looked like at any point in time
- RTO (Recovery Time Objective) drops from **days to minutes**

---
## 10. Exam-Style Practice Questions

---

**Q1: Which statement best describes the declarative approach used by Terraform?**

A) You write step-by-step instructions that Terraform executes in order  
B) You describe the desired end state and Terraform determines the steps to reach it  
C) You write scripts that call cloud APIs directly in sequence  
D) You specify each API call Terraform should make and in what order  

<details>
<summary>Answer</summary>

**Answer: B**  
Terraform is declarative — you describe what you want (desired state), and Terraform's planning engine determines the sequence of API calls needed to achieve it. Options A, C, and D describe imperative approaches.

</details>

---

**Q2: A team runs `terraform apply` on a configuration that matches the current state of their infrastructure. What is the expected result?**

A) Terraform destroys all resources and recreates them  
B) Terraform reports an error because no changes are needed  
C) Terraform makes no changes and outputs "No changes. Your infrastructure matches the configuration."  
D) Terraform creates duplicate resources alongside the existing ones  

<details>
<summary>Answer</summary>

**Answer: C**  
This demonstrates Terraform's idempotency — applying the same configuration multiple times when resources already match produces no changes. Terraform compares desired state to actual state before taking any action.

</details>

---

**Q3: Which of the following is an advantage of using IaC over manual infrastructure provisioning? (Select all that apply)**

A) Infrastructure changes can be reviewed via pull requests before deployment  
B) Infrastructure configuration files automatically eliminate the need for security groups  
C) Environments can be rebuilt quickly from version-controlled configuration  
D) The same workflow and tooling works across multiple cloud providers  
E) Manual configuration through cloud consoles becomes unnecessary for security changes  

<details>
<summary>Answer</summary>

**Answer: A, C, D**  
A = peer review via PRs; C = disaster recovery / repeatable rebuilds; D = multi-cloud agnostic workflow. B is incorrect — IaC still requires security groups to be configured. E is incorrect — IaC does not eliminate the ability to make console changes; it just tracks drift from them.

</details>

---
## 11. Real-World Scenarios

<details>
<summary>Scenario 1 — Multi-Region Disaster Recovery Drill</summary>

**Situation:**
A SaaS company's platform runs entirely in AWS `us-east-1`. The CTO requires a documented DR plan that can restore the environment in `us-west-2` within 30 minutes. Currently, infrastructure was provisioned manually over 18 months — no one has written down the full architecture.

**How IaC solves it:**
The team performs a reverse-engineering exercise using `terraform import` to bring all existing resources under Terraform management. They parameterise the region as a variable.

```hcl
# variables.tf
variable "aws_region" {
  description = "Target AWS region"
  type        = string
  default     = "us-east-1"
}

# provider.tf
provider "aws" {
  region = var.aws_region
}

# main.tf — resources reference var.aws_region through the provider
resource "aws_vpc" "main" {
  cidr_block = "10.0.0.0/16"
  tags       = { Name = "main-vpc-${var.aws_region}" }
}
```

**CLI commands for DR drill:**
```bash
# Create a new workspace for the DR environment
terraform workspace new dr-us-west-2

# Apply to us-west-2 with a variable override
terraform apply -var="aws_region=us-west-2" -auto-approve
```

**Expected outcome:**
- Full environment created in `us-west-2` in under 20 minutes
- DR drill is repeatable — run quarterly without manual effort
- The configuration file is the architecture document

**Exam sub-objective demonstrated:** IaC advantages — disaster recovery speed, repeatability, multi-region consistency.

</details>

<details>
<summary>Scenario 2 — Infrastructure Drift Detected and Corrected</summary>

**Situation:**
An engineer manually adds an inbound rule to a production security group via the AWS console to debug a connectivity issue. They forget to remove it. Two weeks later, a security audit finds the unexpected open port 8080 exposed to `0.0.0.0/0`.

**How IaC detects and corrects drift:**
```hcl
# The correct security group configuration in Terraform
resource "aws_security_group" "app" {
  name = "app-sg"

  ingress {
    from_port   = 443
    to_port     = 443
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }
  # Port 8080 is NOT in the Terraform configuration
}
```

```bash
# Running terraform plan detects the drift
$ terraform plan

  ~ resource "aws_security_group" "app" {
      # (unauthorized port 8080 rule detected as an extra ingress rule)
      - ingress {
          - from_port   = 8080
          - to_port     = 8080
          - protocol    = "tcp"
          - cidr_blocks = ["0.0.0.0/0"]
        }
    }

Plan: 0 to add, 1 to change, 0 to destroy.

# Applying corrects drift immediately
$ terraform apply -auto-approve
```

**Expected outcome:**
- The unauthorized port 8080 rule is removed
- Security group returns to its declared state
- Future drifts are caught at the next `terraform plan` run (in CI or scheduled)

**Exam sub-objective demonstrated:** IaC idempotency, desired state vs actual state reconciliation, peer review preventing manual changes.

</details>

<details>
<summary>Scenario 3 — Promoting an Environment Through Dev → Staging → Production</summary>

**Situation:**
A startup needs identical dev, staging, and production environments but has limited budget. Previously, they manually created each environment and spent 3 days per environment. Configurations inevitably diverged — a bug only reproduced in production because staging was missing a specific IAM policy.

**IaC solution using workspaces and variable files:**

```hcl
# variables.tf
variable "environment" {
  type = string
}

variable "instance_type" {
  type = string
}

# main.tf — same configuration, different sizes per env
resource "aws_instance" "app" {
  ami           = data.aws_ami.ubuntu.id
  instance_type = var.instance_type
  tags          = { Environment = var.environment }
}
```

```hcl
# dev.tfvars
environment   = "dev"
instance_type = "t3.micro"

# staging.tfvars
environment   = "staging"
instance_type = "t3.small"

# production.tfvars
environment   = "production"
instance_type = "m5.large"
```

```bash
# Create dev
terraform workspace new dev
terraform apply -var-file=dev.tfvars

# Create staging
terraform workspace new staging
terraform apply -var-file=staging.tfvars

# Create production
terraform workspace new production
terraform apply -var-file=production.tfvars
```

**Expected outcome:**
- All three environments use identical IAM policies, networking structure, and application configuration
- Only instance sizes differ per environment
- A bug in dev will reproduce in staging and production — no more environment-specific surprises

**Exam sub-objective demonstrated:** IaC consistency across environments, variable precedence, workspace state separation.

</details>

<details>
<summary>Scenario 4 — Hybrid Cloud: On-Premises VMware + AWS in One Workspace</summary>

**Situation:**
A financial services company has a regulatory requirement that customer PII data must stay on-premises. Their application tier can run in AWS, but their database tier must run on VMware vSphere in their data center. They need both tiers managed consistently with audit trails for all changes.

**Hybrid Terraform configuration:**

```hcl
terraform {
  required_providers {
    vsphere = { source = "hashicorp/vsphere", version = "~> 2.4" }
    aws     = { source = "hashicorp/aws",     version = "~> 5.0" }
  }
}

provider "vsphere" {
  vsphere_server = "vcenter.corp.local"
  user           = var.vsphere_user
  password       = var.vsphere_password
  allow_unverified_ssl = false
}

provider "aws" { region = "us-east-1" }

# On-premises database VM
resource "vsphere_virtual_machine" "database" {
  name             = "pgsql-primary-01"
  resource_pool_id = data.vsphere_resource_pool.datacenter.id
  datastore_id     = data.vsphere_datastore.ssd_datastore.id
  num_cpus         = 8
  memory           = 32768
}

# AWS application tier
resource "aws_ecs_service" "app" {
  name            = "app-service"
  cluster         = aws_ecs_cluster.main.id
  desired_count   = 3
  task_definition = aws_ecs_task_definition.app.arn
}

# Direct Connect link between environments
resource "aws_dx_connection" "to_datacenter" {
  name      = "dx-to-datacenter"
  bandwidth = "1Gbps"
  location  = "EqDC2"
}
```

**Expected outcome:**
- All infrastructure changes (both on-prem and cloud) go through the same Git PR process
- Compliance team reviews every change via pull requests before merge
- Full audit trail in Git satisfies regulatory requirements
- Engineers use the same `terraform plan / apply` workflow regardless of whether the target is VMware or AWS

**Exam sub-objective demonstrated:** Multi-cloud and hybrid cloud IaC advantages, service-agnostic provider model.

</details>

<details>
<summary>Scenario 5 — New Engineer Onboards in 30 Minutes Using IaC</summary>

**Situation:**
A startup has 4 engineers and no documentation. When the original lead engineer leaves, the new engineer cannot figure out what infrastructure exists or how to set up a local development environment. There are no wikis, no diagrams, and the AWS console has hundreds of unnamed resources.

**The IaC solution for future-proofing:**

After migrating to IaC, the README includes:

```markdown
## Getting Started

1. Clone this repository
2. Install Terraform >= 1.5
3. Configure AWS credentials (`aws configure`)
4. Run:
   terraform init
   terraform workspace new your-name-dev
   terraform apply -var-file=dev.tfvars
5. Your dev environment is ready. Access URLs are in `terraform output`.
```

The `.tf` files document:
- Every resource that exists (data as documentation)
- How resources relate to each other (dependency graph)
- Why certain decisions were made (comments in HCL)
- Variable defaults that reflect standard configuration

```hcl
# main.tf with inline documentation
resource "aws_elasticache_cluster" "session" {
  # Redis for session storage — auth tokens expire after 24h (see AUTH_TOKEN_TTL)
  # Using t3.micro in dev/staging, r6g.large in production (see instance_type variable)
  cluster_id           = "session-${terraform.workspace}"
  engine               = "redis"
  node_type            = var.cache_instance_type
  num_cache_nodes      = 1
  parameter_group_name = "default.redis7"
  port                 = 6379
}
```

**Expected outcome:**
- New engineer has a working dev environment in 30 minutes
- The `.tf` files are the architecture documentation — always accurate, never stale
- No tribal knowledge required to understand or extend the infrastructure

**Exam sub-objective demonstrated:** IaC as documentation, consistency, and collaboration advantages over manual provisioning.

</details>

---
## Quick-Reference Cheat Sheet

```
IaC Core Concepts for the Exam
═══════════════════════════════════════════════════════

DEFINITION
  IaC = Provisioning infrastructure through machine-readable
        config files instead of manual processes

KEY PRINCIPLES
  Idempotency     → Apply same config N times → same result
  Declarative     → Describe WHAT you want (Terraform, CloudFormation)
  Imperative      → Describe HOW to get there (Bash, Ansible scripts)

TERRAFORM'S APPROACH
  1. Write HCL describing desired state
  2. terraform plan → compute diff (desired vs actual)
  3. terraform apply → execute changes
  4. State file tracks what Terraform manages

PLAN OUTPUT SYMBOLS
  +    create       -    destroy
  ~    update       -/+  destroy and recreate
  <=   read (data source)

ADVANTAGES OVER MANUAL
  Consistency · Repeatability · Version control · Peer review
  Automation · Documentation-as-code · DR speed · Cost visibility

TERRAFORM vs ANSIBLE (common exam question)
  Terraform  → Provision infra (create/destroy cloud resources)
  Ansible    → Configure existing systems (install packages, etc.)
  Terraform  → Declarative + state-tracked
  Ansible    → Imperative + no built-in state

MULTI-CLOUD
  Same init/plan/apply workflow for AWS + Azure + GCP + on-prem
  3,000+ providers on registry.terraform.io
═══════════════════════════════════════════════════════
```